## Regresion Logistica 

Inicialmente probamos con la regresion logistica, un algoritmo diseñado para algorimtos que son linealmentes separables 

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd

def train_logistic_regression(X, y): 
    regularizacion = list(np.logspace(-4, 4, 9))
    rows = []
    for c in regularizacion: 
        accs_train, accs_val, f1s, recalls = [],[],[],[]
        
        kf = KFold(n_splits=5)
        for train_index, val_index in kf.split(X): 
            xtrain, xval = X[train_index], X[val_index]
            ytrain, yval = y[train_index], y[val_index]
            
            scaler = StandardScaler() 
            xtrain, xval = scaler.fit_transform(xtrain), scaler.transform(xval)
            
            model = LogisticRegression(max_iter=1000000, 
                                            tol=1e-6, 
                                            random_state=0,
                                            C=c)

            model.fit(xtrain, ytrain)
            
            predictions_train = model.predict(xtrain)
            predictions_val = model.predict(xval)
            
            accs_train.append(accuracy_score(ytrain, predictions_train))
            accs_val.append(accuracy_score(yval, predictions_val))
            f1s.append(f1_score(yval, predictions_val))
            recalls.append(recall_score(yval, predictions_val))
            

        rows.append({    
            'Regularizacion C': c,
            'media acc train' : np.mean(accs_train),
            'media acc val': np.mean(accs_val), 
            'IC acc val': np.std(accs_val),
            'media f1' : np.mean(f1s),
            'IC f1': np.std(f1s),
            'media recall': np.mean(recalls),
            'IC recall': np.std(recalls)
        }) 

    results = pd.DataFrame(rows)
    return results.sort_values(by='media f1', ascending=False)


In [ ]:
# Con los datos sin filtrado 
from libs.data_configuration import get_data_partitioned

X_train, X_test, y_train, y_test = get_data_partitioned()

train_logistic_regression(X_train, y_train)

,Regularizacion C,media acc train,media acc val,IC acc val,media f1,IC f1,media recall,IC recall
8,10000.0000,0.948875,0.944875,0.005469,0.945079,0.004686,0.951126,0.004020
7,1000.0000,0.948937,0.944375,0.005646,0.944592,0.004893,0.950880,0.004426
6,100.0000,0.948594,0.944250,0.004153,0.944471,0.003286,0.951124,0.003503
5,10.0000,0.948000,0.944250,0.004426,0.944471,0.003436,0.950881,0.003509
4,1.0000,0.947313,0.942250,0.003804,0.942489,0.002860,0.949357,0.003034
3,0.1000,0.944750,0.941250,0.002622,0.941489,0.001622,0.948605,0.004521
2,0.0100,0.938594,0.936500,0.002550,0.936511,0.003001,0.940825,0.004649
1,0.0010,0.926187,0.925500,0.003653,0.924706,0.003747,0.918746,0.006507
0,0.0001,0.899500,0.898875,0.006560,0.894797,0.005637,0.863162,0.016823


In [ ]:
# Para los datos filtrados

from libs.data_configuration import get_filtered_data_partitioned

X_train, X_test, y_train, y_test = get_filtered_data_partitioned()

train_logistic_regression(X_train, y_train)

,Regularizacion C,media acc train,media acc val,IC acc val,media f1,IC f1,media recall,IC recall
7,1000.0000,0.941875,0.938875,0.007939,0.939222,0.008133,0.945229,0.003494
6,100.0000,0.941906,0.938875,0.007939,0.939222,0.008133,0.945229,0.003494
8,10000.0000,0.941906,0.938875,0.007939,0.939222,0.008133,0.945229,0.003494
5,10.0000,0.941812,0.938750,0.008139,0.939111,0.008287,0.945234,0.003756
4,1.0000,0.940375,0.937750,0.008354,0.938071,0.008519,0.943472,0.003435
3,0.1000,0.936656,0.934250,0.008956,0.934629,0.009042,0.940473,0.003812
2,0.0100,0.932031,0.932125,0.008611,0.932462,0.008475,0.937021,0.002171
1,0.0010,0.922563,0.921875,0.008082,0.921572,0.007738,0.917592,0.005877
0,0.0001,0.899156,0.897625,0.008305,0.894272,0.008264,0.865798,0.018085


Se obtienen resultados considerablemente buenos cuando se itera con parametro de regularización $c=100$ 

Aunque es muy probable que según los resultados obtenidos el problema de naturaleza lineal, probaremos con diferentes enfoques más allá de una simple linea

Si usamos como nuestra métrica principal el `f1_score` vemos que nuestro mejor modelo es el que usa el kernel `rbf` y valor $c=10$